In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import cv2
import time
import timm
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import SimpleITK as sitk
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

c:\Users\Edrill-LT\Documents\Projects\Python\Thoracic-Disease-Classifier-ResNet50\.torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
# !! SET YOUR MODEL CHECKPOINT PATHS HERE !!
RNNET_MST_CHECKPOINT  = './rnnet-mst.pth'       # your proposed model
RESNET50_CHECKPOINT   = './resnet50_binary.pth'  # your baseline

TARGET_SIZE  = 224
BATCH_SIZE   = 16
NUM_WORKERS  = 0

split_base_path = './dataset_nodule21/cxr_images/proccessed_data/split_data'
test_images_path = f'{split_base_path}/test/images'
test_csv_path    = f'{split_base_path}/test/metadata_test.csv'

subset_csv_path = './dataset_nodule21/cxr_images/proccessed_data/subset_metadata2.csv'

candidate_image_dirs = [
    f'{split_base_path}/test/images',
    f'{split_base_path}/val/images',
    f'{split_base_path}/train/images',
]

OUTPUT_DIR = './reviewer3_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

val_transform = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def resolve_img_path(img_name):
    for d in candidate_image_dirs:
        p = os.path.join(d, img_name)
        if os.path.exists(p):
            return p
    return None

def load_mha_as_pil(img_path):
    image_itk = sitk.ReadImage(img_path)
    arr = sitk.GetArrayFromImage(image_itk)
    if len(arr.shape) == 3:
        arr = arr[0]
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx > mn:
        arr = ((arr - mn) / (mx - mn) * 255).astype(np.uint8)
    else:
        arr = np.zeros_like(arr, dtype=np.uint8)
    return Image.fromarray(np.stack([arr, arr, arr], axis=-1))

print('✓ Configuration loaded')

✓ Configuration loaded


In [3]:
# ── Model Definitions ─────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=2, mlp_ratio=1.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim), nn.Dropout(dropout)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        if H * W > 196:
            x_down = F.adaptive_avg_pool2d(x, (14, 14))
            H_d, W_d = 14, 14
            x_seq = x_down.flatten(2).transpose(1, 2)
        else:
            x_seq = x.flatten(2).transpose(1, 2)
            H_d, W_d = H, W
        x_norm = self.norm1(x_seq)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x_seq = x_seq + attn_out
        x_seq = x_seq + self.mlp(self.norm2(x_seq))
        x_out = x_seq.transpose(1, 2).reshape(B, C, H_d, W_d)
        if H_d != H or W_d != W:
            x_out = F.interpolate(x_out, size=(H, W), mode='bilinear', align_corners=False)
        return x + x_out


class RNNetMST(nn.Module):
    def __init__(self, num_classes=2, num_heads=2, dropout=0.1):
        super().__init__()
        backbone = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.act1, backbone.maxpool)
        self.stage1 = backbone.layer1
        self.stage2 = backbone.layer2
        self.stage3 = backbone.layer3
        self.stage4 = backbone.layer4
        self.trans1 = TransformerBlock(256,  num_heads=2,        mlp_ratio=1.0, dropout=dropout)
        self.trans2 = TransformerBlock(512,  num_heads=2,        mlp_ratio=1.0, dropout=dropout)
        self.trans3 = TransformerBlock(1024, num_heads=num_heads, mlp_ratio=1.0, dropout=dropout)
        self.trans4 = TransformerBlock(2048, num_heads=num_heads, mlp_ratio=1.0, dropout=dropout)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.trans1(self.stage1(x))
        x = self.trans2(self.stage2(x))
        x = self.trans3(self.stage3(x))
        x = self.trans4(self.stage4(x))
        return self.classifier(self.global_pool(x).flatten(1))


class ResNet50Baseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        backbone = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.act1, backbone.maxpool)
        self.stage1 = backbone.layer1
        self.stage2 = backbone.layer2
        self.stage3 = backbone.layer3
        self.stage4 = backbone.layer4
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        return self.classifier(self.global_pool(x).flatten(1))


def load_rnnet_mst(checkpoint_path, device):
    model = RNNetMST(num_classes=2, num_heads=2, dropout=0.1)
    ckpt  = torch.load(checkpoint_path, map_location='cpu')
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    return model.to(device).eval()


def load_resnet50(checkpoint_path, device):
    model = ResNet50Baseline(num_classes=2)
    ckpt  = torch.load(checkpoint_path, map_location='cpu')
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    # remap keys
    new_state = {}
    for k, v in state.items():
        if k.startswith('conv1'):          new_state['stem.0.' + k] = v
        elif k.startswith('bn1'):          new_state['stem.1.' + k] = v
        elif k.startswith('layer1'):       new_state['stage1.' + k[7:]] = v
        elif k.startswith('layer2'):       new_state['stage2.' + k[7:]] = v
        elif k.startswith('layer3'):       new_state['stage3.' + k[7:]] = v
        elif k.startswith('layer4'):       new_state['stage4.' + k[7:]] = v
        elif k == 'fc.weight':             new_state['classifier.weight'] = v
        elif k == 'fc.bias':               new_state['classifier.bias'] = v
    model.load_state_dict(new_state, strict=False)
    return model.to(device).eval()

print('✓ Model definitions ready')

✓ Model definitions ready


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — Sample images for different nodule sizes (Reviewer 3, Point 1)
# ══════════════════════════════════════════════════════════════════════════════
print('Section 1: Sample images for different nodule sizes')

subset_df = pd.read_csv(subset_csv_path).copy()
subset_df['resolved_path'] = subset_df['img_name'].apply(resolve_img_path)
subset_df = subset_df[subset_df['resolved_path'].notna()].reset_index(drop=True)
subset_df['bbox_area'] = subset_df['width'] * subset_df['height']
subset_df['bbox_max_dim'] = subset_df[['width','height']].max(axis=1)

# Define size categories
small_df  = subset_df[subset_df['bbox_max_dim'] < 40].sort_values('bbox_max_dim')
medium_df = subset_df[(subset_df['bbox_max_dim'] >= 40) & (subset_df['bbox_max_dim'] < 70)].sort_values('bbox_max_dim')
large_df  = subset_df[subset_df['bbox_max_dim'] >= 70].sort_values('bbox_max_dim', ascending=False)

print(f'Small  nodules (<40px)    : {len(small_df)}')
print(f'Medium nodules (40-70px)  : {len(medium_df)}')
print(f'Large  nodules (>=70px)   : {len(large_df)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
categories = [
    ('Small (<40px)',   small_df),
    ('Medium (40-70px)', medium_df),
    ('Large (>=70px)',  large_df),
]

for ax, (title, df) in zip(axes, categories):
    if len(df) == 0:
        ax.set_title(f'{title}\n(no samples)')
        ax.axis('off')
        continue

    row  = df.iloc[0]
    pil  = load_mha_as_pil(row['resolved_path'])
    orig_w, orig_h = pil.size
    pil_resized = pil.resize((TARGET_SIZE, TARGET_SIZE))

    # Scale bbox to resized image
    scale_x = TARGET_SIZE / orig_w
    scale_y = TARGET_SIZE / orig_h
    x = row['x'] * scale_x
    y = row['y'] * scale_y
    w = row['width']  * scale_x
    h = row['height'] * scale_y

    ax.imshow(pil_resized, cmap='gray')
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(f'{title}\nBBox: {int(row["width"])}x{int(row["height"])}px', fontsize=12)
    ax.axis('off')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'nodule_size_samples.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'✓ Saved: {save_path}')

Section 1: Sample images for different nodule sizes
Small  nodules (<40px)    : 28
Medium nodules (40-70px)  : 119
Large  nodules (>=70px)   : 24
✓ Saved: ./reviewer3_figures\nodule_size_samples.png


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — TP, TN, FP, FN sample images (Reviewer 3, Point 2)
# ══════════════════════════════════════════════════════════════════════════════
print('\nSection 2: TP, TN, FP, FN sample images')

# Load RNNet-MST for predictions
rnnet_model = load_rnnet_mst(RNNET_MST_CHECKPOINT, device)
print('✓ RNNet-MST loaded')

# Load test set
test_df = pd.read_csv(test_csv_path).copy()

# Get one unique image per label
test_images_df = test_df.drop_duplicates(subset='img_name').reset_index(drop=True)
test_images_df['resolved_path'] = test_images_df['img_name'].apply(
    lambda x: os.path.join(test_images_path, x) if os.path.exists(os.path.join(test_images_path, x)) else None
)
test_images_df = test_images_df[test_images_df['resolved_path'].notna()].reset_index(drop=True)

# Get predictions for all test images
all_preds  = []
all_labels = []
all_paths  = []

rnnet_model.eval()
with torch.no_grad():
    for _, row in tqdm(test_images_df.iterrows(), total=len(test_images_df), desc='Predicting'):
        pil = load_mha_as_pil(row['resolved_path'])
        img_tensor = val_transform(pil).unsqueeze(0).to(device)
        output = rnnet_model(img_tensor)
        pred = output.argmax(dim=1).item()
        all_preds.append(pred)
        all_labels.append(int(row['label']))
        all_paths.append(row['resolved_path'])

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_paths  = np.array(all_paths)

# Find one example of each case
tp_idx = np.where((all_preds == 1) & (all_labels == 1))[0]
tn_idx = np.where((all_preds == 0) & (all_labels == 0))[0]
fp_idx = np.where((all_preds == 1) & (all_labels == 0))[0]
fn_idx = np.where((all_preds == 0) & (all_labels == 1))[0]

print(f'TP: {len(tp_idx)} | TN: {len(tn_idx)} | FP: {len(fp_idx)} | FN: {len(fn_idx)}')

cases = [
    ('True Positive (TP)\nNodule — Correctly Detected',   tp_idx, 'green'),
    ('True Negative (TN)\nNo Nodule — Correctly Rejected', tn_idx, 'blue'),
    ('False Positive (FP)\nNo Nodule — Incorrectly Flagged', fp_idx, 'orange'),
    ('False Negative (FN)\nNodule — Missed',              fn_idx, 'red'),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for ax, (title, idx_arr, color) in zip(axes, cases):
    if len(idx_arr) == 0:
        ax.set_title(f'{title}\n(none found)')
        ax.axis('off')
        continue

    img_path = all_paths[idx_arr[3]]
    pil = load_mha_as_pil(img_path)
    pil_resized = pil.resize((TARGET_SIZE, TARGET_SIZE))
    ax.imshow(pil_resized, cmap='gray')

    # Draw bbox if positive label
    img_name = os.path.basename(img_path)
    img_rows = test_df[test_df['img_name'] == img_name]
    if all_labels[idx_arr[0]] == 1 and len(img_rows) > 0:
        row = img_rows.iloc[0]
        img_w = row.get('img_width', 1024)
        img_h = row.get('img_height', 1024)
        x = (row['x'] / img_w) * TARGET_SIZE
        y = (row['y'] / img_h) * TARGET_SIZE
        w = (row['width']  / img_w) * TARGET_SIZE
        h = (row['height'] / img_h) * TARGET_SIZE
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(4)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'tp_tn_fp_fn_samples.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'✓ Saved: {save_path}')


Section 2: TP, TN, FP, FN sample images
✓ RNNet-MST loaded


Predicting: 100%|██████████| 733/733 [00:52<00:00, 13.91it/s]


TP: 152 | TN: 545 | FP: 23 | FN: 13
✓ Saved: ./reviewer3_figures\tp_tn_fp_fn_samples.png


In [39]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Computational cost comparison (Reviewer 3, Point 6)
# ══════════════════════════════════════════════════════════════════════════════
# !! SET THE MODEL YOU WANT TO BENCHMARK HERE !!
# Options: 'rnnet_mst' or 'resnet50'
MODEL_TO_TEST = 'rnnet_mst'

print(f'\nSection 3: Computational Cost — {MODEL_TO_TEST}')

if MODEL_TO_TEST == 'rnnet_mst':
    model_gpu = load_rnnet_mst(RNNET_MST_CHECKPOINT, torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
    model_cpu = load_rnnet_mst(RNNET_MST_CHECKPOINT, torch.device('cpu'))
    model_label = 'RNNet-MST'
elif MODEL_TO_TEST == 'resnet50':
    model_gpu = load_resnet50(RESNET50_CHECKPOINT, torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
    model_cpu = load_resnet50(RESNET50_CHECKPOINT, torch.device('cpu'))
    model_label = 'ResNet-50 Baseline'
else:
    raise ValueError(f'Unknown MODEL_TO_TEST: {MODEL_TO_TEST}. Use rnnet_mst or resnet50')

# Count parameters
total_params    = sum(p.numel() for p in model_cpu.parameters())
trainable_params = sum(p.numel() for p in model_cpu.parameters() if p.requires_grad)
print(f'\n{model_label} Parameters:')
print(f'  Total      : {total_params:,}')
print(f'  Trainable  : {trainable_params:,}')

# Warmup + timing function
N_WARMUP = 10
N_RUNS   = 100
dummy_input_gpu = torch.randn(1, 3, TARGET_SIZE, TARGET_SIZE).to(
    torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
)
dummy_input_cpu = torch.randn(1, 3, TARGET_SIZE, TARGET_SIZE)

# GPU timing
if torch.cuda.is_available():
    print(f'\nGPU Timing ({N_RUNS} runs after {N_WARMUP} warmup)...')
    model_gpu.eval()
    with torch.no_grad():
        for _ in range(N_WARMUP):
            _ = model_gpu(dummy_input_gpu)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(N_RUNS):
            _ = model_gpu(dummy_input_gpu)
        torch.cuda.synchronize()
        end = time.perf_counter()
    gpu_time_ms = (end - start) / N_RUNS * 1000
    print(f'  Mean inference time (GPU): {gpu_time_ms:.2f} ms/image')
else:
    gpu_time_ms = None
    print('  GPU not available — skipping GPU timing')

# CPU timing
print(f'\nCPU Timing ({N_RUNS} runs after {N_WARMUP} warmup)...')
model_cpu.eval()
with torch.no_grad():
    for _ in range(N_WARMUP):
        _ = model_cpu(dummy_input_cpu)
    start = time.perf_counter()
    for _ in range(N_RUNS):
        _ = model_cpu(dummy_input_cpu)
    end = time.perf_counter()
cpu_time_ms = (end - start) / N_RUNS * 1000
print(f'  Mean inference time (CPU): {cpu_time_ms:.2f} ms/image')

print('\n' + '='*60)
print(f'COMPUTATIONAL COST SUMMARY — {model_label}')
print('='*60)
print(f'  Total Parameters : {total_params:,}')
if gpu_time_ms:
    print(f'  GPU Inference    : {gpu_time_ms:.2f} ms/image')
print(f'  CPU Inference    : {cpu_time_ms:.2f} ms/image')
print('\nRun this cell again with MODEL_TO_TEST = \'resnet50\' to get baseline numbers.')


Section 3: Computational Cost — rnnet_mst

RNNet-MST Parameters:
  Total      : 56,973,890
  Trainable  : 56,973,890

GPU Timing (100 runs after 10 warmup)...
  Mean inference time (GPU): 11.05 ms/image

CPU Timing (100 runs after 10 warmup)...
  Mean inference time (CPU): 103.52 ms/image

COMPUTATIONAL COST SUMMARY — RNNet-MST
  Total Parameters : 56,973,890
  GPU Inference    : 11.05 ms/image
  CPU Inference    : 103.52 ms/image

Run this cell again with MODEL_TO_TEST = 'resnet50' to get baseline numbers.
